In [ ]:
# Copy code files into current directory from drive.
import os

from deep_qrs_detector import DeepQRSDetector
from deep_qrs_predictor import DeepQRSPredictor
from ecg_dataset_manager import DatasetManager


from ucsd_ecg_dataset import ECGDataset

import numpy as np
import pandas as pd
import biosppy.signals as bsp

# The confusion matrix calculation
import ja_analysis

WANDB_PROJECT_NAME = "ECG Experiments"

In [ ]:
import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))

Set MITBIH_RUN to ***False*** to test on **UCSD** data and to ***True*** to test on **MIT-BIH** Dataset

In [ ]:
# Test on UCSD data (if set to False) 
MITBIH_RUN = True

run_parameters = {}

if(MITBIH_RUN):
    run_parameters["csv_file_name"] = 'mit_bih_dataset.csv'
    run_parameters['EVALUATE_TRIGGER_DETECTIONS'] = False
    
else:
    run_parameters["csv_file_name"] = 'Matched_ECG_MRI_Final.csv' 
    run_parameters['EVALUATE_TRIGGER_DETECTIONS'] = True 

run_parameters['evaluation_type'] = 'test'
run_parameters['EVALUATE_HAMILTON'] = True
run_parameters['EVALUATE_CNN_PREDICTOR'] = True
run_parameters['EVALUATE_CNN_DETECTOR'] = True

run_parameters["DETECTION_CONFIDENCE_THRESHOLD"] = 30.0, # <-- 30.0 datapoints = 120 milliseconds
run_parameters["PEAK_COOLDOWN_THRESHOLD"] = 50.0, # <-- 50.0 datapoints = 200 milliseconds


In [ ]:
DATASET_SAMPLING_RATE = 1000
TARGET_SAMPLING_RATE = 250

UCSD_PROCESSED_DATASET_LOCATION = './data/old_preprocessed/'
MITDB_DATASET_LOCATION = './data/mitdb/raw/'

dm = DatasetManager(UCSD_PROCESSED_DATASET_LOCATION,'',MITDB_DATASET_LOCATION)


def calculateSensitivity(conf_matrix):
    return (conf_matrix['TP'] / (conf_matrix['TP'] + conf_matrix['FN'] + 0.00001))

def calculatePPV(conf_matrix):
    return (conf_matrix['TP'] / (conf_matrix['TP'] + conf_matrix['FP'] + 0.00001))

def calculateFPR(conf_matrix):
    return (conf_matrix["FP"]/(conf_matrix["TN"] + conf_matrix["FP"] + 0.00001))

def calculateSpecificity(conf_matrix):
    return (conf_matrix["TN"] / (conf_matrix["FP"] + conf_matrix["TN"] + 0.00001))

def calculateF1(conf_matrix):
    sensitivity = calculateSensitivity(conf_matrix)
    ppv = calculatePPV(conf_matrix)
    return (2* sensitivity*ppv) / (sensitivity+ppv + 0.00001)


To calculate Metrics for ***BOTH*** MRI_Types **1.5T** and ***3T*** set ***dataset_df*** to ***dataset_both*** otherwise set to either dataset_15 or dataset_3

In [ ]:
# If we are evaluating VCG trigger, get the detections from the dataset
if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
    vcg_ds = ECGDataset("./data/ucsd/") 

cnn_detector = DeepQRSDetector('./models/cnn_detector.h5')
cnn_predictor = DeepQRSPredictor('./models/cnn_predictor.h5',run_parameters['DETECTION_CONFIDENCE_THRESHOLD'],run_parameters['PEAK_COOLDOWN_THRESHOLD'])

dataset_both = pd.read_csv(os.path.join('./data/',run_parameters['csv_file_name']))

if(MITBIH_RUN):
    pass
else:
    dataset_15 = dataset_both[dataset_both['MRI_Type'] == '1.5T']
    dataset_3 = dataset_both[dataset_both['MRI_Type'] == '3T']

dataset_df = dataset_both

allTriggerResults = []
allHamiltonResults = []
allCnnDetectorResults = []
allCnnPredictorResults = []

results_dataframe = pd.DataFrame(columns=["detection_type","TP","TN","FP","FN","jitter","accuracy","ja_score","MAE","offset","offset_ms","loss","sensitivity","specificity","ppv", "fpr", "F1"])

for index, row in dataset_df.iterrows():
    minimum = 0
    maximum = 240000000

    if(row['scan'] is None):
      continue

    if(row['type'] != run_parameters['evaluation_type']):
        continue

    print(f"Processing {row['scan']} {row['scan']}")

    if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
        try:
            trigger_detections = vcg_ds.get_trigger_detections(row['location'],row['timestamp'],row['session_id'])
            trigger_detections = trigger_detections / 4
        except:
            trigger_detections = []
    
    ecg2_downsampled_data, labelstudio_annotations = dm.LoadSignalAndAnnotations(row['scan'], database = row['dataset'])

    labelstudio_annotations.sort()

    if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
        if(len(trigger_detections) == 0):
            print("No trigger detections found for " + row['scan'])
            continue
        ja_result_trigger = ja_analysis.evaluate(np.array(trigger_detections), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"Trigger","TP":ja_result_trigger["TP"],"TN":ja_result_trigger["TN"],"FP":ja_result_trigger["FP"],"FN":ja_result_trigger["FN"],"jitter":ja_result_trigger["ja"],"accuracy":ja_result_trigger["accuracy"],"ja_score":ja_result_trigger["ja"],"MAE":ja_result_trigger["MAE"],"offset":512,"offset_ms":0,'sensitivity': calculateSensitivity(ja_result_trigger),'specificity': calculateSpecificity(ja_result_trigger),'ppv' : calculatePPV(ja_result_trigger), 'fpr' : calculateFPR(ja_result_trigger), 'F1' : calculateF1(ja_result_trigger)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)

        allTriggerResults.append(ja_result_trigger)

    if(run_parameters['EVALUATE_HAMILTON']):
        biosspy_peaks = bsp.ecg.hamilton_segmenter(ecg2_downsampled_data, sampling_rate=250)
        biosspy_peaks = biosspy_peaks[0]

        biosspy_peaks.sort()

        biosppy_results = ja_analysis.evaluate(np.array(biosspy_peaks), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"Hamilton","TP":biosppy_results["TP"],"TN":biosppy_results["TN"],"FP":biosppy_results["FP"],"FN":biosppy_results["FN"],"jitter":biosppy_results["ja"],"accuracy":biosppy_results["accuracy"],"ja_score":biosppy_results["ja"],"MAE":biosppy_results["MAE"],"offset":512,"offset_ms":0,'sensitivity': calculateSensitivity(biosppy_results),'specificity': calculateSpecificity(biosppy_results),'ppv' : calculatePPV(biosppy_results), 'fpr' : calculateFPR(biosppy_results), 'F1' : calculateF1(biosppy_results)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)
        allHamiltonResults.append(biosppy_results)

    if(run_parameters['EVALUATE_CNN_DETECTOR']):
        # Detect R peaks
        cnn_detector_r_peaks,mean_detection_offset = cnn_detector.detect_peaks(ecg2_downsampled_data)


        # Sort the cnn_detector_r_peaks list
        cnn_detector_r_peaks.sort()

        ja_result_cnn_detector = ja_analysis.evaluate(np.array(cnn_detector_r_peaks), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"CNN Detector","TP":ja_result_cnn_detector["TP"],"TN":ja_result_cnn_detector["TN"],"FP":ja_result_cnn_detector["FP"],"FN":ja_result_cnn_detector["FN"],"jitter":ja_result_cnn_detector["ja"],"accuracy":ja_result_cnn_detector["accuracy"],"ja_score":ja_result_cnn_detector["ja"],"MAE":ja_result_cnn_detector["MAE"],"offset":mean_detection_offset,"offset_ms":4.0*(mean_detection_offset-512.0),'sensitivity': calculateSensitivity(ja_result_cnn_detector),'specificity': calculateSpecificity(ja_result_cnn_detector),'ppv' : calculatePPV(ja_result_cnn_detector), 'fpr' : calculateFPR(ja_result_cnn_detector), 'F1' : calculateF1(ja_result_cnn_detector)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)

        allCnnDetectorResults.append(ja_result_cnn_detector)

    if(run_parameters['EVALUATE_CNN_PREDICTOR']):
        # Detect R peaks
        cnn_predictor_r_peaks,mean_prediction_offset = cnn_predictor.detect_peaks(ecg2_downsampled_data)


        # Sort the cnn_predictor_r_peaks list
        cnn_predictor_r_peaks.sort()

        ja_result_cnn_predictor = ja_analysis.evaluate(np.array(cnn_predictor_r_peaks), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"CNN Predictor","TP":ja_result_cnn_predictor["TP"],"TN":ja_result_cnn_predictor["TN"],"FP":ja_result_cnn_predictor["FP"],"FN":ja_result_cnn_predictor["FN"],"jitter":ja_result_cnn_predictor["ja"],"accuracy":ja_result_cnn_predictor["accuracy"],"ja_score":ja_result_cnn_predictor["ja"],"MAE":ja_result_cnn_predictor["MAE"],"offset":mean_prediction_offset,"offset_ms":4.0*(mean_prediction_offset-512.0),'sensitivity': calculateSensitivity(ja_result_cnn_predictor),'specificity': calculateSpecificity(ja_result_cnn_predictor),'ppv' : calculatePPV(ja_result_cnn_predictor), 'fpr' : calculateFPR(ja_result_cnn_predictor), 'F1' : calculateF1(ja_result_cnn_predictor)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)

        allCnnPredictorResults.append(ja_result_cnn_predictor)


# Print the results
if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
    trigger_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'Trigger']['accuracy'].mean()
    trigger_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'Trigger']['ja_score'].mean()
    trigger_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'Trigger']['F1'].mean()

    print("Trigger Detection Results: {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(trigger_average_f1*100, trigger_average_accuracy * 100, trigger_average_jitter * 1000,trigger_average_jitter))
if(run_parameters['EVALUATE_HAMILTON']):
    hamilton_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'Hamilton']['accuracy'].mean()
    hamilton_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'Hamilton']['ja_score'].mean()
    hamilton_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'Hamilton']['F1'].mean()

    print("Hamilton Detection Results: {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(hamilton_average_f1 * 100,hamilton_average_accuracy * 100, hamilton_average_jitter * 1000,hamilton_average_jitter))


ecg_detector_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Detector']['accuracy'].mean()
ecg_detector_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Detector']['ja_score'].mean()
ecg_detector_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Detector']['F1'].mean()

print("CNN Detector Results:  {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(ecg_detector_average_f1 * 100,ecg_detector_average_accuracy * 100, ecg_detector_average_jitter * 1000,ecg_detector_average_jitter))

ecg_predictor_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Predictor']['accuracy'].mean()
ecg_predictor_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Predictor']['ja_score'].mean()
ecg_predictor_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Predictor']['F1'].mean()

print("CNN Predictor Results:  {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(ecg_predictor_average_f1 * 100,ecg_predictor_average_accuracy * 100, ecg_predictor_average_jitter * 1000,ecg_predictor_average_jitter))

# Save the results to a CSV file
results_dataframe.to_csv("./results/evaluation_results.csv")

In [ ]:
results_dataframe.groupby('detection_type')[['accuracy','fpr','specificity','ppv','sensitivity','F1','MAE']].mean()

In [ ]:
results_dataframe.groupby('detection_type')[['accuracy','fpr','specificity','ppv','sensitivity','F1','MAE']].count()